In [75]:
import os
from pathlib import Path
from dotenv import load_dotenv
from rich import print
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage
from langchain.chat_models import init_chat_model
import langgraph
from langgraph.graph import StateGraph, START, END


## Calling LLM

In [78]:
load_dotenv()
llm = os.getenv("GROQ_MODEL", "openai/gpt-oss-20b")
llm

'openai/gpt-oss-20b'

In [80]:
class State(TypedDict):
    """State of the graph."""
    question: str
    answer: str

In [81]:
def answer_node(state: State) -> dict:
    response = llm.invoke(state["question"])
    return {"answer": response.content}


def build_graph():
    graph = StateGraph(State)

    graph.add_node("answer", answer_node)
    graph.add_edge(START, "answer")
    graph.add_edge("answer", END)

    return graph.compile()

In [92]:
def main():
    agent = build_graph()
    agent_result = agent.invoke({"question": "What is the capital of France?"})
    print(agent_result["answer"])